# The Gradio App

The last step is a web UI. Design choices for this app:

- **On-demand, not scheduled** — one button. Press it, and one autonomous
  planning run happens live: scan the shop, judge the candidates, surface
  the best bargain. A visitor sees the whole thing work in about a couple
  of minutes.
- **The `AutonomousPlanningAgent` drives** — the LLM-with-tools planner, so
  the streamed logs show a model calling other models as tools.
- **The logs are the show** — streamed into the page, color-coded per agent.
  Alongside them: a verdict card for the latest find and a table of every
  bargain found so far (persistent memory).

In [ ]:
import gradio as gr

from utils.listings import Listing, Opportunity

### Piece 1: just the header

In [ ]:
with gr.Blocks(title="Wine Value Hunter", fill_width=True) as ui:
    gr.Markdown(
        '<div style="text-align: center; font-size: 24px"><strong>Wine Value Hunter'
        "</strong> — agentic AI that finds VFM wines in a live shop</div>"
    )

ui.launch(inbrowser=True)

### Piece 2: the results table, wired to sample data

In [ ]:
sample = Opportunity(
    listing=Listing(
        title="2019 River Vibes Pinot Noir [WE92]",
        tasting_note="Concentrated, with boysenberry, black plum and baking spices.",
        points=92,
        price=7.99,
        url="https://bottlebarn.com/products/2019-river-vibes-pinot-noir",
        variety="Pinot Noir",
        country="United States",
        province="California",
        region="Mendocino",
        winery="River Vibes",
    ),
    estimated_vfm=51,
    actual_vfm=81,
    delta=30,
)


def table_for(opps):
    return [
        [
            opp.listing.title,
            opp.listing.points,
            f"${opp.listing.price:.2f}",
            opp.estimated_vfm,
            opp.actual_vfm,
            f"{opp.delta:+d}",
            opp.listing.url,
        ]
        for opp in opps
    ]


with gr.Blocks(title="Wine Value Hunter", fill_width=True) as ui:
    gr.Markdown(
        '<div style="text-align: center; font-size: 24px"><strong>Wine Value Hunter'
        "</strong></div>"
    )
    results = gr.Dataframe(
        value=table_for([sample]),
        headers=["Wine", "Points", "Price", "Typical VFM", "VFM at price", "Delta", "URL"],
        wrap=True,
    )

ui.launch(inbrowser=True)

### The real thing lives in `app.py`

Everything else — the button wiring, the background thread, streaming the
agents' colored logs into the page, the verdict card — is in
[`app.py`](app.py). The interesting mechanics:

- The button starts the framework run **in a thread**, so the UI stays alive.
- A logging handler pushes every agent log line into a **queue**; a generator
  drains the queue and yields updated HTML, so the page updates live.
- ANSI color codes in the logs are converted to HTML spans, keeping each
  agent's color from the terminal.
- Memory persists in `memory.json` — every press only surfaces NEW wines,
  and the table accumulates across runs.

In [ ]:
# Optional: wipe the find history to start fresh
from pathlib import Path

Path("memory.json").unlink(missing_ok=True)

# Running the final product

One press ≈ one to three minutes (more if the Modal container is cold) and
roughly $0.10-0.25 of API usage: one scanner extraction, ~5 ensemble
estimates (each a frontier call + a Modal call + the local network), and
the planner's own loop.

In [ ]:
!uv run app.py